# Titanic Survival Prediction

Refactor aligned to `Titanic_plan.md`.

1. EDA on train and test
2. Leakage-safe feature engineering
3. Pipeline-based model training with CV
4. Final evaluation and error analysis
5. Kaggle submission generation
6. Submission review notes


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


## 1. EDA

Load both datasets, inspect dtypes, missing values, class balance, and basic distributions. No row drops are needed.


In [ ]:
train = pd.read_csv('Datasets/train.csv')
test = pd.read_csv('Datasets/test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print('No row drops are needed.')

display(train.head())

eda = pd.DataFrame({
    'train_dtype': train.dtypes.astype(str),
    'train_missing': train.isna().sum(),
    'test_dtype': test.dtypes.astype(str),
    'test_missing': test.isna().sum(),
})
display(eda)

class_balance = train['Survived'].value_counts().rename(index={0: 'not survived', 1: 'survived'})
display(class_balance.to_frame('count'))
display((train['Survived'].value_counts(normalize=True).sort_index()).rename('share').to_frame())


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

sns.countplot(data=train, x='Pclass', hue='Survived', ax=axes[0])
axes[0].set_title('Pclass vs Survived')

sns.countplot(data=train, x='Sex', hue='Survived', ax=axes[1])
axes[1].set_title('Sex vs Survived')

sns.countplot(data=train, x='Embarked', hue='Survived', ax=axes[2])
axes[2].set_title('Embarked vs Survived')

sns.countplot(data=train, x='SibSp', ax=axes[3])
axes[3].set_title('SibSp distribution')

sns.countplot(data=train, x='Parch', ax=axes[4])
axes[4].set_title('Parch distribution')

sns.histplot(data=train, x='Age', hue='Survived', bins=30, kde=True, ax=axes[5])
axes[5].set_title('Age distribution')

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=train, x='Fare', hue='Survived', bins=30, kde=True, ax=axes[0])
axes[0].set_title('Fare distribution')
sns.boxplot(data=train, x='Survived', y='Fare', ax=axes[1])
axes[1].set_title('Fare vs Survived')
plt.tight_layout()
plt.show()


## 2. Feature Engineering

Leakage note: test data is used only for domain-driven feature construction at prediction time. All learned preprocessing, including age and fare imputations, is fit on train folds only inside the pipeline.


In [ ]:
def simplify_title(name):
    title = name.str.extract(r' ([A-Za-z]+)\.', expand=False)
    title = title.replace({
        'Mlle': 'Miss',
        'Ms': 'Miss',
        'Mme': 'Mrs',
        'Lady': 'Rare',
        'Countess': 'Rare',
        'Capt': 'Rare',
        'Col': 'Rare',
        'Don': 'Rare',
        'Dr': 'Rare',
        'Major': 'Rare',
        'Rev': 'Rare',
        'Sir': 'Rare',
        'Jonkheer': 'Rare',
        'Dona': 'Rare',
    })
    return title.where(title.isin(['Mr', 'Mrs', 'Miss', 'Master']), 'Rare').fillna('Rare')


class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df = pd.DataFrame(X).copy()
        title = simplify_title(df['Name'])
        age_frame = pd.DataFrame({'Title': title, 'Age': df['Age']})
        fare_frame = pd.DataFrame({'Pclass': df['Pclass'], 'Fare': df['Fare']})

        self.age_medians_ = age_frame.groupby('Title')['Age'].median().to_dict()
        self.fare_medians_ = fare_frame.groupby('Pclass')['Fare'].median().to_dict()
        self.age_fallback_ = df['Age'].median()
        self.fare_fallback_ = df['Fare'].median()
        return self

    def transform(self, X):
        df = pd.DataFrame(X).copy()
        df['Title'] = simplify_title(df['Name'])
        df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
        df['HasCabin'] = df['Cabin'].notna().astype(int)
        df['Sex'] = df['Sex'].map({'female': 0, 'male': 1}).fillna(0).astype(int)
        df['PclassOrd'] = (4 - df['Pclass']).astype(int)
        df['Embarked_C'] = df['Embarked'].eq('C').fillna(False).astype(int)

        age = df['Age'].copy()
        for title_name, median_age in self.age_medians_.items():
            mask = age.isna() & (df['Title'] == title_name)
            age.loc[mask] = median_age
        df['Age'] = age.fillna(self.age_fallback_)

        fare = df['Fare'].copy()
        for pclass, median_fare in self.fare_medians_.items():
            mask = fare.isna() & (df['Pclass'] == pclass)
            fare.loc[mask] = median_fare
        df['FareLog'] = np.log1p(fare.fillna(self.fare_fallback_))

        return df[['PclassOrd', 'Title', 'Age', 'FareLog', 'FamilySize', 'HasCabin', 'Sex', 'Embarked_C']]


feature_preview = TitanicFeatureEngineer().fit_transform(train.drop(columns=['Survived']))
display(feature_preview.head())


In [ ]:
baseline_age = train['Age'].fillna(train['Age'].median())
baseline_pred = ((train['Sex'] == 'female') | (baseline_age < 16)).astype(int)

baseline_metrics = {
    'Model': 'Baseline',
    'CV F1': np.nan,
    'OOF Accuracy': accuracy_score(train['Survived'], baseline_pred),
    'OOF Precision': precision_score(train['Survived'], baseline_pred, zero_division=0),
    'OOF Recall': recall_score(train['Survived'], baseline_pred, zero_division=0),
    'OOF F1': f1_score(train['Survived'], baseline_pred, zero_division=0),
    'Best Params': '-',
}

baseline_cm = confusion_matrix(train['Survived'], baseline_pred)
print(classification_report(train['Survived'], baseline_pred, zero_division=0))

plt.figure(figsize=(5, 4))
sns.heatmap(baseline_cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Baseline: women and children first')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


## 3. Model Training

Compare Logistic Regression, Random Forest, and Gradient Boosting using 5-fold stratified CV, `GridSearchCV`, and F1 as the selection metric. Preprocessing lives inside the pipeline so it is fit only on train folds.


In [ ]:
X = train.drop(columns=['Survived'])
y = train['Survived'].copy()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

numeric_features = ['PclassOrd', 'Age', 'FareLog', 'FamilySize', 'HasCabin', 'Sex', 'Embarked_C']
categorical_features = ['Title']

def make_pipeline(model, scale_numeric):
    numeric_step = StandardScaler() if scale_numeric else 'passthrough'
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_step, numeric_features),
            ('title', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ]
    )
    return Pipeline([
        ('features', TitanicFeatureEngineer()),
        ('preprocess', preprocessor),
        ('model', model),
    ])

model_specs = {
    'Logistic Regression': {
        'estimator': LogisticRegression(max_iter=4000, random_state=42),
        'scale_numeric': True,
        'grid': {
            'model__C': [0.3, 1.0, 3.0],
            'model__solver': ['liblinear'],
            'model__penalty': ['l1', 'l2'],
        },
    },
    'Random Forest': {
        'estimator': RandomForestClassifier(random_state=42),
        'scale_numeric': False,
        'grid': {
            'model__n_estimators': [200],
            'model__max_depth': [4, 6, None],
            'model__min_samples_split': [2, 5],
            'model__min_samples_leaf': [1, 2],
        },
    },
    'Gradient Boosting': {
        'estimator': GradientBoostingClassifier(random_state=42),
        'scale_numeric': False,
        'grid': {
            'model__n_estimators': [100, 150],
            'model__learning_rate': [0.03, 0.05],
            'model__max_depth': [2, 3],
            'model__subsample': [0.8, 1.0],
        },
    },
}

searches = {}
oof_predictions = {}
model_rows = []

for name, spec in model_specs.items():
    pipeline = make_pipeline(spec['estimator'], spec['scale_numeric'])
    search = GridSearchCV(pipeline, spec['grid'], cv=cv, scoring='f1', n_jobs=-1)
    search.fit(X, y)
    oof = cross_val_predict(search.best_estimator_, X, y, cv=cv, n_jobs=-1)

    searches[name] = search
    oof_predictions[name] = oof
    model_rows.append({
        'Model': name,
        'CV F1': search.best_score_,
        'OOF Accuracy': accuracy_score(y, oof),
        'OOF Precision': precision_score(y, oof, zero_division=0),
        'OOF Recall': recall_score(y, oof, zero_division=0),
        'OOF F1': f1_score(y, oof, zero_division=0),
        'Best Params': search.best_params_,
    })
    print(f'{name}: {search.best_params_}')

results = pd.DataFrame([baseline_metrics] + model_rows)
results['Delta F1 vs Baseline'] = results['OOF F1'] - baseline_metrics['OOF F1']
display(results.sort_values('OOF F1', ascending=False).reset_index(drop=True).round(3))


In [ ]:
comparison = results.set_index('Model')[['OOF Accuracy', 'OOF Precision', 'OOF Recall', 'OOF F1']]
ax = comparison.plot(kind='bar', figsize=(12, 5))
ax.set_ylim(0, 1)
ax.set_title('Model comparison')
ax.set_ylabel('Score')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()


## 4. Final Evaluation

Select the best model by CV F1, retrain it on the full training set, inspect OOF errors, and plot feature importance. The strongest signals are usually `Title`, `Sex`, `PclassOrd`, `FareLog`, and `FamilySize`. The model still struggles with borderline age cases and sparse or unusual passenger profiles.


In [ ]:
model_only = results[results['Model'] != 'Baseline'].copy()
best_model_name = model_only.sort_values('CV F1', ascending=False).iloc[0]['Model']
best_search = searches[best_model_name]
final_model = best_search.best_estimator_
final_model.fit(X, y)

best_oof = oof_predictions[best_model_name]
cm_final = confusion_matrix(y, best_oof)
tn, fp, fn, tp = cm_final.ravel()

print(f'Best model: {best_model_name}')
print(classification_report(y, best_oof, zero_division=0))
print(f'False positives: {fp}')
print(f'False negatives: {fn}')

plt.figure(figsize=(5, 4))
sns.heatmap(cm_final, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'OOF confusion matrix: {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


In [ ]:
feature_names = final_model.named_steps['preprocess'].get_feature_names_out()
model_step = final_model.named_steps['model']

if hasattr(model_step, 'feature_importances_'):
    importances = pd.Series(model_step.feature_importances_, index=feature_names).sort_values(ascending=False)
    title = f'{best_model_name} feature importance'
elif hasattr(model_step, 'coef_'):
    importances = pd.Series(np.abs(model_step.coef_[0]), index=feature_names).sort_values(ascending=False)
    title = f'{best_model_name} coefficient magnitude'
else:
    perm = permutation_importance(final_model, X, y, n_repeats=10, random_state=42, scoring='f1')
    importances = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
    title = f'{best_model_name} permutation importance'

display(importances.head(10).to_frame('importance'))

plt.figure(figsize=(10, 5))
sns.barplot(x=importances.head(10).values, y=importances.head(10).index, color='#4c78a8')
plt.title(title)
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


## 5. Final Prediction and Submission

Predict the Kaggle test set, preview the submission, run sanity checks, and save `submission_title_family_hascabin_cv.csv`.


In [ ]:
test_predictions = final_model.predict(test)
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_predictions.astype(int),
})

display(submission.head())
print(f'Submission shape: {submission.shape}')
print(submission['Survived'].value_counts().sort_index())
print(f"Unique prediction values: {sorted(submission['Survived'].unique().tolist())}")

assert submission.shape == (len(test), 2)
assert submission['Survived'].isin([0, 1]).all()

submission_file = 'submission_title_family_hascabin_cv.csv'
submission.to_csv(submission_file, index=False)
print(f'Saved {submission_file}')


## 6. Submission Review

Compare this file against earlier Kaggle attempts outside the notebook. If the score is still weak, iterate only on leakage-safe feature engineering or pipeline tuning.
